## Set root

In [1]:
import os
import pyrootutils
from pathlib import Path

root = pyrootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
root_use_case = root / "use_case"

%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Users\juliaq\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\re\_casefix.py' for module 're._casefix': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Julia_Projects\safetycage_library_test\examples\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Julia_Projects\safetycage_library_test\examples\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\juliaq\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib.

## Load data handler and predictions
Then the predictions are loaded into the data handler. These will be packaged together with the other data samples.

In [2]:
import numpy as np
from use_case.src.data.keras_cifar10_datahandler import KerasCIFAR10DataHandler

data_handler = KerasCIFAR10DataHandler(
    data_dir=root_use_case / "data/cifar10"
).from_joblib()

# data_handler.plot_samples()


Data loaded from C:\Julia_Projects\safetycage_library_test\examples\use_case\data\cifar10\cifar10.npz.


# Load prediction model
You can choose between an MLP and a CNN.

In [3]:
from keras.models import load_model

model = load_model(root_use_case / "model/cnn")

model.summary()




Model: "cnn"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape_1 (Reshape)         (None, 32, 32, 3)         0         
                                                                 
 conv2d_3 (Conv2D)           (None, 30, 30, 32)        896       
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 15, 15, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_4 (Conv2D)           (None, 13, 13, 64)        18496     
                                                                 
 max_pooling2d_3 (MaxPoolin  (None, 6, 6, 64)          0         
 g2D)                                                            
                                                                 
 conv2d_5 (Conv2D)           (None, 4, 4, 64)          36928

## Instantiate the model handler.
The model handler acts as a translator between the model and safetycage, translating the syntax of the model to something standardize that the safetycage can use, for all models.

In [4]:
from use_case.src.model.keras_modelhandler import ModelHandlerKeras


model_handler = ModelHandlerKeras(
    model=model, selected_layers=["dense_3"], use_onehot_encoder=True
)

In [5]:
inputs_train, labels_train = data_handler.data_train

predictions_train = model_handler._get_predictions(inputs_train)

1250/1250 [==============================] - 6s 4ms/step


In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy = accuracy_score(labels_train, predictions_train)
precision = precision_score(labels_train, predictions_train, average="weighted")
recall = recall_score(labels_train, predictions_train, average="weighted")

print("Performance on training data:")
print(f"Accuracy: {accuracy * 100:.4f} %")
print(f"Precision: {precision * 100:.4f} %")
print(f"Recall: {recall * 100:.4f} %")

Performance on training data:
Accuracy: 80.5725 %
Precision: 81.6918 %
Recall: 80.5725 %


## Instantiate a safetycage

In [7]:
from components.mahalanobis import Mahalanobis

safetycage = Mahalanobis(model_handler, data_handler, use_preactivations = True, empirical = False,
                         cauchy_weights_per_layer = [], test_type_between_layers = "cauchy", test_type_within_layer = "chi2", alpha = "null")

## Train the safetycage

In [8]:
# Train the safety cage on training data
safetycage.train_cage(x=inputs_train, y=labels_train, y_pred=predictions_train)

## Safetycage predicts on validation data
Safetycage predicts whether samples have been misclassified by classifier. The type of statistic will vary depending on the safetycage method, but will say something about how likely it is a given sample has been misclassified.

In [9]:
inputs_val, labels_val = data_handler.data_val

predictions_val = model_handler._get_predictions(inputs_val)
incorrect_predictions_val = ~(
    np.argmax(predictions_val, axis=1) == np.argmax(labels_val, axis=1)
)

max_probs_val = safetycage.predict(
    x=inputs_val,
    y=predictions_val,
)

313/313 [==============================] - 1s 4ms/step


## Find threshold that maximizes a performance metric
Use the statistics from the validation data produced by the safetycage, to find a theshold that will maximize a metric, when compared to the ground truth (correct or incorrect classification)

In [10]:
from safetycage_testing.utils.evaluate import find_best_threshold, MCC

# Find optimal alpha for specified max metric based on the validation data
optimisation_result = find_best_threshold(
    y_probs=max_probs_val,
    y_true=incorrect_predictions_val,
    metric_fn=MCC,
    greater_is_better=True,
)

threshold_val = optimisation_result["alpha_opt"]
metric_val = optimisation_result["metric_max"]

print("\nOptimal threshold and performance metric based on validation data:")
print(f"Threshold = {threshold_val:.4f}")
print(f"{MCC.__name__} = {metric_val:.4f}")

safetycage.alpha = threshold_val
safetycage.threshold_metric_val = metric_val


Optimal threshold and performance metric based on validation data:
Threshold = 0.8805
MCC = 0.1165


## Test with optimal threshold, the performance on the test data partition.
Finally we test the safetycage with the inferred threshold parameter, on a hold out test data set. The flags are the predicitons made by the safetycage.

In [11]:
inputs_test, labels_test = data_handler.data_test

prediction_test = model_handler._get_predictions(inputs_test)
incorrect_prediction_test = ~(
    np.argmax(prediction_test, axis=1) == np.argmax(labels_test, axis=1)
)

max_probs_test = safetycage.predict(
    x=inputs_test,
    y=prediction_test,
)

flags_test = safetycage.flag(max_probs_test)

313/313 [==============================] - 2s 5ms/step


# Calculate metrics of the predictions on the test data set.
Compute a set of metrics that will inform us of the safetycages abillity to flag misclassification made by a certain model on a certain dataset.

In [12]:
import json
import os

# from src.utils.evaluate import (
#     calculate_confusion_rates,
#     calculate_metrics,
#     calculate_auroc,
# )

from safetycage_testing.utils.evaluate import (
    calculate_confusion_rates,
    calculate_metrics,
    calculate_auroc,
)

# Calculate confusion rates and metrics based on test data
confusion_rates_test = calculate_confusion_rates(
    y=incorrect_prediction_test,
    y_pred=flags_test,
)

# Calculate AUROC based on test data
auroc_test = calculate_auroc(
    safetycage=safetycage,
    y_true=incorrect_prediction_test,
    y_scores=max_probs_test,  # Assuming the first column contains the scores
)

# Calculate metrics based on test data
metrics_test = calculate_metrics(y=incorrect_prediction_test, y_pred=flags_test)

# Add confusion rates and AUROC to metrics dictionary
metrics_test.update(confusion_rates_test)
metrics_test.update({"AUROC": auroc_test})

with open("./results/metrics_test.json", "w") as f:
    json.dump(metrics_test, f, indent=4)

for name, value in metrics_test.items():
    print(f"{name}: {value:.4f}")

Precision: 0.3620
Recall: 0.8986
Specificity: 0.1975
NPV: 0.7936
MCC: 0.1223
TP: 3022.0000
TN: 1311.0000
FP: 5326.0000
FN: 341.0000
AUROC: 0.5754


## Plots
Further we plot a number of figures showing further the performance of safetycage. These plots are the 
1. Confusion matrix, 
2. Threshold-metric curve (how the  metric varies with the threshold)
3. Reciever-operator curve.

In [13]:
# from src.utils.evaluate import calculate_roc_curve

# from src.utils.plot_functions import (
#     plot_alpha_metric_curve,
#     plot_confusion_matrix,
#     plot_roc_curve,
# )

from safetycage_testing.utils.evaluate import calculate_roc_curve

from safetycage_testing.utils.plot_functions import (
    plot_alpha_metric_curve,
    plot_confusion_matrix,
    plot_roc_curve,
)

best_metric_dict = find_best_threshold(
    y_probs=max_probs_test,
    y_true=incorrect_prediction_test,
    metric_fn=MCC,
    greater_is_better=True,
)

# plots
plot_alpha_metric_curve(
    **best_metric_dict,
    thresholds=optimisation_result["alphas"],
    scores=optimisation_result["metric_values"],
    output_path="./results",
    alpha_val=safetycage.alpha,
    metric_val=safetycage.threshold_metric_val,
    val_label_offset=(0.0, 0.10) # the validation label tends to cover the data
)

plot_confusion_matrix(
    y_true=incorrect_prediction_test,
    y_pred=flags_test,
    normalize=None,
    output_path="./results",
)

roc = calculate_roc_curve(
    safetycage=safetycage,
    y_true=incorrect_prediction_test,
    statistics=max_probs_test,
    num_thresholds=1e4,
)

plot_roc_curve(
    **roc,
    output_path="./results"
)